 ## PySpark Set Up

In [1]:
from pyspark.sql import SparkSession

# Start SparkSession - our engine
spark = SparkSession.builder \
    .appName("LogiScale_Phase1_DataCo") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .getOrCreate()
# clean output
spark.sparkContext.setLogLevel("ERROR") 

print("Spark version:", spark.version)
print("SparkSession started successfully!")

Spark version: 4.1.1
SparkSession started successfully!


 ## Load the  CSV file into PySpark

In [2]:
# Load the dataset
df_raw = spark.read.csv(
    "DataCoSupplyChainDataset.csv",
    header=True,
    inferSchema=True,
    encoding="ISO-8859-1"
)

print(f"Raw row count  : {df_raw.count():,}")
print(f"Raw col count  : {len(df_raw.columns)}")
df_raw.printSchema()

Raw row count  : 180,519
Raw col count  : 53
root
 |-- Type: string (nullable = true)
 |-- Days for shipping (real): integer (nullable = true)
 |-- Days for shipment (scheduled): integer (nullable = true)
 |-- Benefit per order: double (nullable = true)
 |-- Sales per customer: double (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: integer (nullable = true)
 |-- Category Id: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Customer City: string (nullable = true)
 |-- Customer Country: string (nullable = true)
 |-- Customer Email: string (nullable = true)
 |-- Customer Fname: string (nullable = true)
 |-- Customer Id: integer (nullable = true)
 |-- Customer Lname: string (nullable = true)
 |-- Customer Password: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Customer State: string (nullable = true)
 |-- Customer Street: string (nullable = true)
 |-- Customer Zipcode: integer (nullable = t

## Drop Unnecessary Columns

In [3]:
# These columns are irrelevant to logistics/supply chain analysis
cols_to_drop = [
    "Customer Email",
    "Customer Password",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Customer Zipcode",
    "Customer City",
    "Customer State",
    "Product Description",
    "Product Image",
    "Order Zipcode",
    "Order Item Cardprod Id",
    "Product Card Id",
    "Product Category Id",
    "Order Item Id",
    "Order Customer Id",
    "Product Status"
]

df_clean = df_raw.drop(*cols_to_drop)

print(f"Columns after drop : {len(df_clean.columns)}")
df_clean.printSchema()

Columns after drop : 36
root
 |-- Type: string (nullable = true)
 |-- Days for shipping (real): integer (nullable = true)
 |-- Days for shipment (scheduled): integer (nullable = true)
 |-- Benefit per order: double (nullable = true)
 |-- Sales per customer: double (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: integer (nullable = true)
 |-- Category Id: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Customer Country: string (nullable = true)
 |-- Customer Id: integer (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Department Id: integer (nullable = true)
 |-- Department Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Market: string (nullable = true)
 |-- Order City: string (nullable = true)
 |-- Order Country: string (nullable = true)
 |-- order date (DateOrders): string (nullable = true)
 |-- Order Id: integer (nullable = t

## Rename Columns to Match Project Names

In [4]:
# Rename every column to clean project-standard names
df = df_clean \
    .withColumnRenamed("Type",                          "payment_type") \
    .withColumnRenamed("Days for shipping (real)",      "ata_days") \
    .withColumnRenamed("Days for shipment (scheduled)", "eta_days") \
    .withColumnRenamed("Benefit per order",             "benefit_per_order") \
    .withColumnRenamed("Sales per customer",            "sales_per_customer") \
    .withColumnRenamed("Delivery Status",               "delivery_status") \
    .withColumnRenamed("Late_delivery_risk",            "late_delivery_risk") \
    .withColumnRenamed("Category Id",                   "category_id") \
    .withColumnRenamed("Category Name",                 "category_name") \
    .withColumnRenamed("Customer Country",              "customer_country") \
    .withColumnRenamed("Customer Id",                   "customer_id") \
    .withColumnRenamed("Customer Segment",              "customer_segment") \
    .withColumnRenamed("Department Id",                 "warehouse_id") \
    .withColumnRenamed("Department Name",               "warehouse_name") \
    .withColumnRenamed("Latitude",                      "latitude") \
    .withColumnRenamed("Longitude",                     "longitude") \
    .withColumnRenamed("Market",                        "market") \
    .withColumnRenamed("Order City",                    "order_city") \
    .withColumnRenamed("Order Country",                 "order_country") \
    .withColumnRenamed("order date (DateOrders)",       "order_date") \
    .withColumnRenamed("Order Id",                      "order_id") \
    .withColumnRenamed("Order Item Discount",           "item_discount") \
    .withColumnRenamed("Order Item Discount Rate",      "item_discount_rate") \
    .withColumnRenamed("Order Item Product Price",      "product_price") \
    .withColumnRenamed("Order Item Profit Ratio",       "profit_ratio") \
    .withColumnRenamed("Order Item Quantity",           "units_sold") \
    .withColumnRenamed("Sales",                         "sales") \
    .withColumnRenamed("Order Item Total",              "order_total") \
    .withColumnRenamed("Order Profit Per Order",        "order_profit") \
    .withColumnRenamed("Order Region",                  "route") \
    .withColumnRenamed("Order State",                   "order_state") \
    .withColumnRenamed("Order Status",                  "order_status") \
    .withColumnRenamed("Product Name",                  "sku_name") \
    .withColumnRenamed("Product Price",                 "sku_price") \
    .withColumnRenamed("shipping date (DateOrders)",    "ship_date") \
    .withColumnRenamed("Shipping Mode",                 "shipping_mode")

print("Column renaming done.")
df.printSchema()

Column renaming done.
root
 |-- payment_type: string (nullable = true)
 |-- ata_days: integer (nullable = true)
 |-- eta_days: integer (nullable = true)
 |-- benefit_per_order: double (nullable = true)
 |-- sales_per_customer: double (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- late_delivery_risk: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- warehouse_id: integer (nullable = true)
 |-- warehouse_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- market: string (nullable = true)
 |-- order_city: string (nullable = true)
 |-- order_country: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- item_discount: double (nullable = tr

 ## Fix Date Columns & Calculate Delay

In [5]:
from pyspark.sql.functions import (
    to_timestamp, col, when,
    round as spark_round
)

# Fix date columns (DataCo uses M/d/yyyy H:mm format)
df = df \
    .withColumn("order_date", to_timestamp(col("order_date"), "M/d/yyyy H:mm")) \
    .withColumn("ship_date",  to_timestamp(col("ship_date"),  "M/d/yyyy H:mm"))

# Calculate delay in days: actual - scheduled
# Positive = Late, Negative = Early, Zero = On Time
df = df.withColumn(
    "delay_days",
    spark_round(col("ata_days") - col("eta_days"), 2)
).withColumn(
    "delivery_flag",
    when(col("delay_days") > 0, "Late")
    .when(col("delay_days") < 0, "Early")
    .otherwise("On Time")
)

print("Date parsing and delay calculation done.")
df.show(5)

Date parsing and delay calculation done.
+------------+--------+--------+-----------------+------------------+----------------+------------------+-----------+--------------+----------------+-----------+----------------+------------+--------------+-----------+------------+------------+----------+-------------+-------------------+--------+-------------+------------------+-------------+------------+----------+------+-----------+------------+--------------+---------------+---------------+------------+---------+-------------------+--------------+----------+-------------+
|payment_type|ata_days|eta_days|benefit_per_order|sales_per_customer| delivery_status|late_delivery_risk|category_id| category_name|customer_country|customer_id|customer_segment|warehouse_id|warehouse_name|   latitude|   longitude|      market|order_city|order_country|         order_date|order_id|item_discount|item_discount_rate|product_price|profit_ratio|units_sold| sales|order_total|order_profit|         route|    order_s

In [6]:
# Date parsing and delay calculation (top 5 rows)
from pyspark.sql.functions import to_timestamp, col, when

# Convert date columns to proper timestamp
df = df.withColumn(
        "order_date",
        to_timestamp(col("order_date"), "M/d/yyyy H:mm")
     ).withColumn(
        "ship_date",
        to_timestamp(col("ship_date"), "M/d/yyyy H:mm")
     )

# Create delay column (no need to round since it's days)
df = df.withColumn(
    "delay_days",
    col("ata_days") - col("eta_days")
)

# Create delivery status flag
df = df.withColumn(
    "delivery_flag",
    when(col("delay_days") > 0, "Late")
    .when(col("delay_days") < 0, "Early")
    .otherwise("On Time")
)

# Show sample output
print("Date parsing and delay calculation done.")
df.select("ata_days", "eta_days", "delay_days", "delivery_flag").show(5)

Date parsing and delay calculation done.
+--------+--------+----------+-------------+
|ata_days|eta_days|delay_days|delivery_flag|
+--------+--------+----------+-------------+
|       3|       4|        -1|        Early|
|       5|       4|         1|         Late|
|       4|       4|         0|      On Time|
|       3|       4|        -1|        Early|
|       2|       4|        -2|        Early|
+--------+--------+----------+-------------+
only showing top 5 rows


##  Performance Benchmark: Pandas vs PySpark

In [7]:
import time
import pandas as pd

FILE = "DataCoSupplyChainDataset.csv"

# ══════════════════════════════════════════
# PANDAS BENCHMARK
# ══════════════════════════════════════════
print("Running Pandas benchmark...")
t0 = time.time()

pdf = pd.read_csv(FILE, encoding="ISO-8859-1")
pdf["delay_days"] = (
    pdf["Days for shipping (real)"] - pdf["Days for shipment (scheduled)"]
)
pandas_result = pdf.groupby("Order Region")["delay_days"].mean()

pandas_time = round(time.time() - t0, 2)
pandas_memory = round(pdf.memory_usage(deep=True).sum() / 1e6, 1)

print(f"Pandas  → Time: {pandas_time}s | Memory: {pandas_memory} MB")

# ══════════════════════════════════════════
# PYSPARK BENCHMARK
# ══════════════════════════════════════════
print("\nRunning PySpark benchmark...")
from pyspark.sql.functions import avg

t0 = time.time()

spark_result = df.groupBy("route").agg(
    avg("delay_days").alias("avg_delay")
)
spark_result.collect()   # .collect() forces full execution

spark_time = round(time.time() - t0, 2)

print(f"PySpark → Time: {spark_time}s | Memory: distributed (no RAM limit)")

# ══════════════════════════════════════════
# PRINT COMPARISON
# ══════════════════════════════════════════
print("\n" + "="*45)
print("   BENCHMARK RESULT — PHASE 1")
print("="*45)
print(f"   Dataset rows       : {df.count():,}")
print(f"   Pandas load time   : {pandas_time}s")
print(f"   PySpark exec time  : {spark_time}s")
print(f"   Pandas RAM used    : {pandas_memory} MB")
print(f"   PySpark RAM used   : No limit (distributed)")
print("="*45)
print("   Conclusion: At 10x data size, Pandas")
print("   crashes. PySpark scales infinitely.")
print("="*45)

Running Pandas benchmark...
Pandas  → Time: 2.82s | Memory: 350.2 MB

Running PySpark benchmark...
PySpark → Time: 1.22s | Memory: distributed (no RAM limit)

   BENCHMARK RESULT — PHASE 1
   Dataset rows       : 180,519
   Pandas load time   : 2.82s
   PySpark exec time  : 1.22s
   Pandas RAM used    : 350.2 MB
   PySpark RAM used   : No limit (distributed)
   Conclusion: At 10x data size, Pandas
   crashes. PySpark scales infinitely.


## Save the Cleaned DataFrame 

In [8]:
# Save cleaned data 
df.toPandas().to_csv("logiscale_cleaned.csv", index=False)

print("Cleaned file saved as: logiscale_cleaned.csv")
print("Cleaned file saved successfully.")
print(f"Final columns : {len(df.columns)}")
print(f"Final rows    : {df.count():,}")
print("\nFinal column list:")
for c in df.columns:
    print(f"  - {c}")

Cleaned file saved as: logiscale_cleaned.csv
Cleaned file saved successfully.
Final columns : 38
Final rows    : 180,519

Final column list:
  - payment_type
  - ata_days
  - eta_days
  - benefit_per_order
  - sales_per_customer
  - delivery_status
  - late_delivery_risk
  - category_id
  - category_name
  - customer_country
  - customer_id
  - customer_segment
  - warehouse_id
  - warehouse_name
  - latitude
  - longitude
  - market
  - order_city
  - order_country
  - order_date
  - order_id
  - item_discount
  - item_discount_rate
  - product_price
  - profit_ratio
  - units_sold
  - sales
  - order_total
  - order_profit
  - route
  - order_state
  - order_status
  - sku_name
  - sku_price
  - ship_date
  - shipping_mode
  - delay_days
  - delivery_flag
